**# 0) 환경 준비**
- Google Drive 마운트
- 기본 경로/폴더 준비
- 라이브러리 임포트 & 시드 고정


In [ ]:
# ── Google Drive Mount
import os, sys
try:
    from google.colab import drive  # Colab 환경
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive', force_remount=False)
    print("Drive OK:", os.path.exists("/content/drive/MyDrive"))
except Exception as e:
    print("Colab이 아니거나, 드라이브 마운트 예외:", e)

# ── Paths
BASE_DIR      = "/content/drive/MyDrive/RIID_CNN"
DATA_DIR      = f"{BASE_DIR}/Dataset"     # 원본 .txt 파일 폴더
PROCESSED_DIR = f"{BASE_DIR}/processed"   # 전처리 산출물
MODELS_DIR    = f"{PROCESSED_DIR}/models"
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# ── Imports
import re, json, math, time, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, f1_score

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore")

# ── Reproducibility
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = bool(torch.cuda.is_available())
print("device:", device, "| pin_memory:", PIN_MEMORY)


Mounted at /content/drive
Drive OK: True
device: cpu | pin_memory: False


**# 1) 파서 & 전처리 (Strict)**
- 파일명 → (model, sensor, serial)
- 한 줄(line) → (timestamp, Count, GC, Temperature(raw), Spc[2048])
- 섭씨온도 변환: temp_c = (Temperature * 330)/4096 - 60
- 라벨 규칙: sum(Spc)/Count ≤ 500 cps → BKG, 그 외 → Cs-137
- **정규화 패치**:
  - SQRT 변환(분산 안정화): x ← sqrt(x + 3/8) (선택)
  - NORM_MODE="chan_z": 학습(train-split)에서 **채널별 mean/std**로 Z-score
  - (대안) "per_spec_z": 한 스펙트럼 내부 평균/표준편차 기준 표준화(구버전)
- **물리 게이트**(선택): Cs 대표 피크(≈662keV) 채널 위치 창(window) 밖이면 확률 억제


In [ ]:
# ── 전역 옵션 (필요 시 조정)
NORM_MODE = "chan_z"     # "chan_z"(권장) 또는 "per_spec_z"(기존)
SQRT_TRANSFORM = True    # √변환(Anscombe 유사) on/off
APPLY_PEAK_GATE = True   # Cs 피크 창 기반 억제 on/off
MIN_PEAK_TOL = 8         # 피크창 반폭 하한(채널)

# --- normalization guards ---
STD_FLOOR     = 0.25   # 채널별 표준편차 하한
Z_CLIP        = 8.0    # z-score 클리핑 범위
OCC_THRESH    = 0.01   # 채널 점유율(>0 cps 비율) 하한
MEAN_CPS_FLOOR = 1e-4  # 평균 cps가 너무 작은 채널은 희소로 간주

# ===== OOD(미학습 분포) 거절 설정 =====
PROTO_PCT     = 97.5   # 학습 분포에서 허용 거리 상한(백분위) → 이보다 멀면 Others
MIN_SAMPLES_PER_CLASS = 50   # 프로토타입 계산을 위한 최소 학습 샘플 수

# tri-class 확률 임계(형상만 사용)
TRI_CLASS       = True
TRI_POS_DEFAULT = 0.80   # p>= → Cs-137
TRI_NEG_DEFAULT = 0.20   # p<= → BKG
TRI_MARGIN      = 0.15   # 저장된 best_threshold가 있을 때의 완충폭
APPLY_PEAK_GATE = False  # 피크 게이트를 쓰려면 True(형상만 쓰려면 False 유지)



# ── 라벨 맵
id_to_label = {0: "BKG", 1: "Cs-137"}
label_to_id = {v:k for k,v in id_to_label.items()}

# ── 유틸
def temp_to_celsius(raw_temp):
    # 섭씨온도 = (Temperature*330)/4096 - 60
    return (float(raw_temp) * 330.0) / 4096.0 - 60.0

def minmax_scale(v, vmin, vmax):
    if vmax == vmin: return 0.0
    return (float(v) - vmin) / (vmax - vmin)

def _safe_div(a, b):
    b = float(b) if np.isscalar(b) else np.asarray(b, dtype=np.float32)
    return a / np.where(b==0, 1.0, b)

def _anscombe(x):
    return np.sqrt(x + 3.0/8.0) if SQRT_TRANSFORM else x

# ── 파일명 파서
def parse_filename(fname: str):
    """
    model: "hwSpc_940..." → HH300 / "hwSpc_PR2..." → SPRD
    serial: 'hwSpc_' 과 '(' 사이의 문자열
    sensor: 괄호 안 문자열 (예: CLLBC/CLYC/CSI)
    """
    bn = Path(fname).name
    m = re.match(r"hwSpc_([^()]+)\(([^()]+)\)\.txt$", bn)
    if not m:
        raise ValueError(f"Unexpected filename: {bn}")
    serial = m.group(1)
    sensor = m.group(2)
    if serial.startswith("940"):
        model = "HH300"
    elif serial.startswith("PR2"):
        model = "SPRD"
    else:
        model = "UNKNOWN"
    return model, sensor, serial

# ── 라인 파서 (예시 형식 엄격히)
LINE_RE = re.compile(
    r"^\[(?P<ts>[\d\-:.\s]+)\]\[BackgroundThread\]: \[Nc\] Count: (?P<count>\d+), GC: (?P<gc>\d+), Temperature: (?P<temp>\d+), Spc: (?P<spc>.+)$"
)

def parse_line_strict_auto(line: str):
    """형식이 다르면 None 반환, Spc 채널 수가 2048이 아니면 None."""
    m = LINE_RE.match(line.strip())
    if not m:
        return None
    ts = m.group("ts").strip()
    count = int(m.group("count"))
    gc    = int(m.group("gc"))
    temp  = int(m.group("temp"))
    try:
        spc = np.fromstring(m.group("spc"), sep=",", dtype=np.float32)
    except Exception:
        return None
    if spc.size != 2048:
        return None
    return ts, count, gc, temp, spc

def make_label_from_spc(spc, count):
    cps = float(spc.sum()) / max(1, float(count))
    return ("BKG" if cps <= 500.0 else "Cs-137"), cps


**# 2) 그룹 저장**
- DATA_DIR 아래의 .txt 전부 스캔 → (model, sensor, serial) 분류
- 유효 행만 파싱(2048 채널만) → 메타 + 스펙트럼 누적
- 그룹별(model/sensor/label)로 저장: meta(.parquet/.csv), spc(.npy)
- 인덱스 표 생성(group_index.csv)


In [ ]:
def build_and_save_datasets(data_dir=DATA_DIR, out_dir=PROCESSED_DIR):
    records = []  # group summary
    bank = {}     # (model, sensor, label) -> list of (meta_row, spc)

    txts = sorted([str(p) for p in Path(data_dir).glob("*.txt")])
    if not txts:
        print("No txt files found in", data_dir)
        return None

    kept_all = 0; skipped_all = 0
    for fp in txts:
        model, sensor, serial = parse_filename(fp)
        with open(fp, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                parsed = parse_line_strict_auto(line)
                if not parsed:
                    skipped_all += 1; continue
                ts, count, gc, temp_raw, spc_raw = parsed
                temp_c = temp_to_celsius(temp_raw)
                label_str, cps_val = make_label_from_spc(spc_raw, count)
                key = (model, sensor, label_str)
                bank.setdefault(key, {"meta": [], "spc": []})
                bank[key]["meta"].append({
                    "timestamp": ts, "model": model, "sensor": sensor, "serial": serial,
                    "count": count, "gc": gc, "temp_c": temp_c, "cps": cps_val,
                    "label_str": label_str, "label_id": label_to_id[label_str],
                })
                bank[key]["spc"].append(spc_raw.astype(np.float32))
                kept_all += 1

    print(f"[*] 파싱 완료: 사용 {kept_all}행, 제외 {skipped_all}행")

    rows_idx = []
    for (model, sensor, label), pack in bank.items():
        meta = pd.DataFrame(pack["meta"])
        spc  = np.stack(pack["spc"], axis=0).astype(np.float32)

        stem = f"group_{model}_{sensor}_{label}"
        meta_parquet = f"{out_dir}/{stem}.parquet"
        meta_csv     = f"{out_dir}/{stem}.csv"
        spc_npy      = f"{out_dir}/{stem}.npy"

        meta.to_parquet(meta_parquet, index=False)
        meta.to_csv(meta_csv, index=False)
        np.save(spc_npy, spc)

        rows_idx.append({
            "model": model, "sensor": sensor, "label": label, "n": len(meta),
            "meta_parquet": meta_parquet, "meta_csv": meta_csv, "spc_npy": spc_npy
        })

    df_index = pd.DataFrame(rows_idx).sort_values(["model","sensor","label"]).reset_index(drop=True)
    df_index.to_csv(f"{out_dir}/group_index.csv", index=False)
    print(df_index)
    return df_index

# 실행 (필요시 1회)
df_index = build_and_save_datasets(DATA_DIR, PROCESSED_DIR)


[*] 파싱 완료: 사용 26983행, 제외 4행
   model sensor   label     n  \
0  HH300  CLLBC     BKG   469   
1  HH300  CLLBC  Cs-137  3259   
2  HH300   CLYC     BKG  1475   
3  HH300   CLYC  Cs-137  8262   
4   SPRD  CLLBC     BKG  2335   
5   SPRD  CLLBC  Cs-137  6677   
6   SPRD    CSI     BKG    88   
7   SPRD    CSI  Cs-137  4418   

                                        meta_parquet  \
0  /content/drive/MyDrive/RIID_CNN/processed/grou...   
1  /content/drive/MyDrive/RIID_CNN/processed/grou...   
2  /content/drive/MyDrive/RIID_CNN/processed/grou...   
3  /content/drive/MyDrive/RIID_CNN/processed/grou...   
4  /content/drive/MyDrive/RIID_CNN/processed/grou...   
5  /content/drive/MyDrive/RIID_CNN/processed/grou...   
6  /content/drive/MyDrive/RIID_CNN/processed/grou...   
7  /content/drive/MyDrive/RIID_CNN/processed/grou...   

                                            meta_csv  \
0  /content/drive/MyDrive/RIID_CNN/processed/grou...   
1  /content/drive/MyDrive/RIID_CNN/processed/grou...   
2

**# 3) 채널별 통계/피크창 + Dataset/CNN**
- Train split에서 채널별 mean/std(=chan_z) 저장
- 양성(Cs-137)에서 피크 채널 중앙값 & 허용 반폭 계산 (물리 게이트)
- Dataset은 정규화(normalize_spectrum) 후 텐서 반환
- CNN1D(+aux: temp, gc)


In [ ]:
# === 로드 유틸 ===
def load_group_arrays(model, sensor):
    meta = pd.read_csv(f"{PROCESSED_DIR}/group_{model}_{sensor}_BKG.csv")
    spc0 = np.load(f"{PROCESSED_DIR}/group_{model}_{sensor}_BKG.npy")
    meta1 = pd.read_csv(f"{PROCESSED_DIR}/group_{model}_{sensor}_Cs-137.csv")
    spc1  = np.load(f"{PROCESSED_DIR}/group_{model}_{sensor}_Cs-137.npy")
    meta_all = pd.concat([meta, meta1], ignore_index=True)
    spc_all  = np.concatenate([spc0, spc1], axis=0).astype(np.float32)
    # 라벨은 이미 label_id에 있음
    return meta_all, spc_all

def compute_aux_minmax(meta_tr):
    return {
        "temp_c": {"min": float(meta_tr["temp_c"].min()), "max": float(meta_tr["temp_c"].max())},
        "gc":     {"min": float(meta_tr["gc"].min()),     "max": float(meta_tr["gc"].max())},
    }

def compute_channel_stats(meta, spc, idx):
    counts = meta.iloc[idx]["count"].to_numpy(np.float32)
    X_cps = spc[idx].astype(np.float32) / counts[:, None]   # cps
    X = _anscombe(X_cps)                                    # √변환(선택)
    m = X.mean(axis=0)
    s = X.std(axis=0)
    # 채널 점유율(>0 cps 비율)로 희소 채널 판별
    occ = (X_cps > 0).mean(axis=0)
    valid_mask = (occ >= OCC_THRESH) | (m > MEAN_CPS_FLOOR)
    # 희소 채널은 정규화 영향 줄이기: std 최소값 보장, 비유효 채널은 1.0로 둠
    s = np.where(valid_mask, np.maximum(s, STD_FLOOR), 1.0)
    return m.astype(np.float32), s.astype(np.float32), valid_mask.astype(bool)

def normalize_spectrum(x_counts, count, chan_stats=None):
    x_cps = x_counts.astype(np.float32) / float(count)  # cps
    x = _anscombe(x_cps)
    if NORM_MODE == "chan_z" and chan_stats is not None:
        m = chan_stats["mean"]; s = chan_stats["std"]
        x = (x - m) / np.maximum(s, STD_FLOOR)
        mask = chan_stats.get("valid_mask", None)
        if mask is not None:
            x[~mask] = 0.0          # 희소 채널은 0으로 무시
        x = np.clip(x, -Z_CLIP, Z_CLIP)  # 안전 클리핑
    else:
        mu, sd = float(x.mean()), float(x.std()); sd = (sd if sd>0 else 1.0)
        x = (x - mu) / sd
    return x

def compute_cs_peak_window(meta, spc, idx, labels, chan_stats):
    pos_idx = [i for i in idx if int(labels[i]) == 1]
    if len(pos_idx) < 10:
        return None
    peaks = []
    for i in pos_idx:
        x = normalize_spectrum(spc[i], meta.iloc[i]["count"], chan_stats)
        peaks.append(int(np.argmax(x)))
    peaks = np.array(peaks)
    center = int(np.median(peaks))
    q1, q3 = np.percentile(peaks, [25, 75])
    tol = max(MIN_PEAK_TOL, int((q3 - q1) * 0.75))
    return {"center": center, "tol": tol}

# === Dataset
class GroupSpectrumDataset(Dataset):
    def __init__(self, meta, spc, indices, scaler, chan_stats=None):
        self.meta = meta.reset_index(drop=True)
        self.spc  = spc
        self.idx  = np.array(indices, dtype=np.int64)
        self.scaler = scaler
        self.chan_stats = chan_stats

    def __len__(self): return len(self.idx)

    def __getitem__(self, k):
        i = int(self.idx[k])
        row = self.meta.iloc[i]
        x_counts = self.spc[i]
        x = normalize_spectrum(x_counts, row["count"], self.chan_stats)
        t = minmax_scale(row["temp_c"], self.scaler["temp_c"]["min"], self.scaler["temp_c"]["max"])
        g = minmax_scale(row["gc"],     self.scaler["gc"]["min"],     self.scaler["gc"]["max"])
        y = int(row["label_id"])
        return torch.from_numpy(x[None,:]).float(), torch.tensor(y).long(), torch.tensor([t,g], dtype=torch.float32)

# === CNN
class CNN1D(nn.Module):
    def __init__(self, use_aux=True):
        super().__init__()
        self.use_aux = use_aux
        self.feat = nn.Sequential(
            nn.Conv1d(1, 16, 9, padding=4), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(16,32,9, padding=4), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32,64,9, padding=4), nn.ReLU(), nn.AdaptiveAvgPool1d(32),
        )
        self.head = nn.Sequential(
            nn.Linear(64*32 + (2 if use_aux else 0), 128), nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, x, aux=None):
        h = self.feat(x).flatten(1)
        if self.use_aux and aux is not None:
            h = torch.cat([h, aux], dim=1)
        logit = self.head(h).view(-1,1)
        return logit, h

# ─────────────────────────────────────────────────────────
# v2↔v1 체크포인트 호환 로더 (run once, before inference)
# ─────────────────────────────────────────────────────────
#import torch
#import torch.nn as nn

# 구버전(CNN1D_V1) 구조: 예전 체크포인트와 호환
class CNN1D_V1(nn.Module):
    """
    과거 모델과의 호환 버전:
      feat: Conv1d(1,32,9,pad=4)->BN->ReLU->Pool
            Conv1d(32,64,9,pad=4)->BN->ReLU->Pool
            Conv1d(64,128,9,pad=4)->BN->ReLU->AdaptiveAvgPool1d(2)
      head: Linear(256(+aux2),256)->ReLU->Dropout(0.2)->Linear(256,1)
    """
    def __init__(self, use_aux=True):
        super().__init__()
        self.use_aux = use_aux
        self.feat = nn.Sequential(
            nn.Conv1d(1, 32, 9, padding=4),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, 9, padding=4),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, 9, padding=4),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool1d(2),   # -> (N,128,2)
        )
        in_dim = 128*2 + (2 if self.use_aux else 0)  # 256 + aux(2)
        self.head = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
        )

    def forward(self, x, aux=None):
        h = self.feat(x).flatten(1)
        if self.use_aux and aux is not None:
            h = torch.cat([h, aux], dim=1)
        logit = self.head(h).view(-1,1)
        return logit, h

def _try_load_state(net, ckpt_path, map_location="cpu", strict=True):
    try:
        sd = torch.load(ckpt_path, map_location=map_location)
        if isinstance(sd, dict) and "state_dict" in sd:
            sd = sd["state_dict"]
        net.load_state_dict(sd, strict=strict)
        return True, None
    except Exception as e:
        return False, e

def load_cnn_from_ckpt_with_fallback(ckpt_path, device):
    """
    1) 현재 노트북의 CNN1D(v2) 구조로 로드 시도
    2) 실패하면 CNN1D_V1(구버전) 구조로 재시도
    """
    # v2 시도 (노트북에 CNN1D가 정의돼 있어야 함)
    if "CNN1D" in globals():
        try:
            net_v2 = CNN1D(use_aux=True).to(device)
            ok, err = _try_load_state(net_v2, ckpt_path, map_location=device, strict=True)
            if ok:
                return net_v2, "v2"
            ok, err2 = _try_load_state(net_v2, ckpt_path, map_location=device, strict=False)
            if ok:
                print("[WARN] v2(strict=False)로 일부 키 무시하고 로드했습니다.")
                return net_v2, "v2"
            else:
                print("[INFO] v2 로드 실패 → v1로 재시도 :", err)
        except Exception as e:
            print("[INFO] v2 인스턴스 생성 실패 → v1로 재시도 :", e)
    else:
        print("[INFO] CNN1D(v2)가 현재 세션에 정의되어 있지 않아 v1로 바로 시도합니다.")

    # v1 시도
    net_v1 = CNN1D_V1(use_aux=True).to(device)
    ok, err = _try_load_state(net_v1, ckpt_path, map_location=device, strict=True)
    if ok:
        print("[INFO] 구버전(v1) 모델 구조로 로드했습니다.")
        return net_v1, "v1"
    ok, err2 = _try_load_state(net_v1, ckpt_path, map_location=device, strict=False)
    if ok:
        print("[WARN] v1(strict=False)로 일부 키 무시하고 로드했습니다.")
        return net_v1, "v1"

    raise RuntimeError(
        "체크포인트를 어떤 구조로도 불러오지 못했습니다.\n"
        f"- path: {ckpt_path}\n"
        f"- Last errors:\n  v1: {err2}"
    )

**# 4) 학습(train_one_group)**
- 시리얼 기준 GroupShuffleSplit (train:val=8:2)
- pos_weight로 클래스 불균형 보정
- best.pt / history.csv / scaler.json 저장
  - scaler.json에 chan mean/std, aux min-max, peak_gate, best_threshold, 옵션 저장


In [ ]:
def safe_name(s):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(s))

def train_one_group(
    model_name, sensor,                       # 'SPRD', 'CSI' 등
    group_row,                                # groups_df.iloc[idx] 같은 한 행(dict/Series)
    epochs=20, batch_size=256, lr=3e-4,
    weight_decay=1e-4, patience=5, num_workers=2,
    seed=42
):
    """
    group_row에는 meta_parquet/meta_csv/spc_npy 경로와 n(샘플 수)이 포함되어 있어야 합니다.
    """
    torch.manual_seed(seed)

    # ----- [1] 저장 경로 먼저 정의 (에러 원인 fix)
    save_dir = os.path.join(PROCESSED_DIR, "models", model_name, sensor)
    os.makedirs(save_dir, exist_ok=True)
    best_path = os.path.join(save_dir, "best.pt")
    last_path = os.path.join(save_dir, "last.pt")
    log_path  = os.path.join(save_dir, "train_log.json")

    # ----- [2] 데이터셋/로더 준비 (사용중인 Dataset 클래스로 구성)
    dset_train, dset_val = make_train_val_datasets_for_group(group_row)  # 기존에 쓰던 함수
    train_loader = DataLoader(dset_train, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True, drop_last=False)
    val_loader   = DataLoader(dset_val,   batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True, drop_last=False)

    # ----- [3] 모델/손실/최적화
    net = CNN1D(use_aux=True).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optim = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=weight_decay)

    best_val = float("inf")
    best_epoch = -1
    no_improve = 0
    history = {"epoch":[],"train_loss":[],"val_loss":[],"val_acc":[]}

    # ----- [4] 학습 루프
    for epoch in range(1, epochs+1):
        net.train()
        run_loss, run_correct, run_total = 0.0, 0, 0
        for xb, auxb, yb in train_loader:   # xb:(B,1,2048) auxb:(B,2) yb:(B,)  ← 중요: yb shape
            xb = xb.to(device)
            auxb = auxb.to(device)
            yb = yb.float().to(device)          # (B,)
            optim.zero_grad()
            logit, _ = net(xb, auxb)            # logit:(B,1)
            logit = logit.squeeze(1)            # (B,)  ← BCEWithLogitsLoss와 호환
            loss = criterion(logit, yb)
            loss.backward()
            optim.step()

            run_loss += loss.item() * xb.size(0)
            pred = (torch.sigmoid(logit) > 0.5).long()
            run_correct += (pred == yb.long()).sum().item()
            run_total += xb.size(0)

        tr_loss = run_loss / max(1, run_total)
        tr_acc  = run_correct / max(1, run_total)

        # ----- [5] 검증
        net.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0
        with torch.no_grad():
            for xb, auxb, yb in val_loader:
                xb = xb.to(device); auxb = auxb.to(device); yb = yb.float().to(device)
                logit, _ = net(xb, auxb)        # (B,1)
                logit = logit.squeeze(1)        # (B,)
                loss = criterion(logit, yb)
                v_loss += loss.item() * xb.size(0)
                pred = (torch.sigmoid(logit) > 0.5).long()
                v_correct += (pred == yb.long()).sum().item()
                v_total += xb.size(0)
        va_loss = v_loss / max(1, v_total)
        va_acc  = v_correct / max(1, v_total)

        history["epoch"].append(epoch)
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)

        # ----- [6] 체크포인트 (여기서 best_path/last_path 사용)
        torch.save(net.state_dict(), last_path)
        if va_loss < best_val:
            best_val = va_loss
            best_epoch = epoch
            torch.save(net.state_dict(), best_path)
            no_improve = 0
        else:
            no_improve += 1

        print(f"[{model_name}/{sensor}] epoch {epoch:02d}/{epochs} "
              f"train {tr_loss:.4f}|{tr_acc:.3f}  val {va_loss:.4f}|{va_acc:.3f} "
              f"{'*' if epoch==best_epoch else ''}")

        if no_improve >= patience:
            print(f"Early stop @ {epoch}, best epoch={best_epoch}, best val={best_val:.4f}")
            break

    # ----- [7] 로그 저장
    with open(log_path, "w", encoding="utf-8") as f:
        json.dump({
            "model": model_name, "sensor": sensor,
            "best_epoch": best_epoch, "best_val": best_val,
            "history": history,
            "best_path": best_path, "last_path": last_path
        }, f, ensure_ascii=False, indent=2)

    return best_path, history

# ── 보조: 대각 마할라노비스 거리
def _mahalanobis_diag_batch(H: np.ndarray, mu: np.ndarray, std: np.ndarray):
    Z = (H - mu[None, :]) / np.maximum(std[None, :], 1e-6)
    return np.sqrt((Z * Z).sum(axis=1))

# ── 보조: (학습셋 인덱스 tr_idx)에서 임베딩 추출 → 클래스별 proto 저장
def compute_class_prototypes(net, meta, spc, tr_idx, chan_stats, scaler, device):
    net.eval()
    feats, labels = [], []
    with torch.no_grad():
        for i in tr_idx:
            row = meta.iloc[i]
            x_norm = normalize_spectrum(spc[i], row["count"], chan_stats)
            t_s = minmax_scale(row["temp_c"], scaler["temp_c"]["min"], scaler["temp_c"]["max"])
            g_s = minmax_scale(row["gc"],     scaler["gc"]["min"],     scaler["gc"]["max"])
            X   = torch.from_numpy(x_norm[None,None,:]).float().to(device)
            AUX = torch.tensor([[t_s, g_s]], dtype=torch.float32, device=device)
            _, h = net(X, AUX)
            feats.append(h.cpu().numpy())
            labels.append(int(row["label"]))
    H = np.concatenate(feats, axis=0)
    Y = np.array(labels, dtype=int)

    proto = {}
    for c in (0, 1):  # 0=BKG, 1=Cs-137
        Hc = H[Y == c]
        if len(Hc) < MIN_SAMPLES_PER_CLASS:
            continue
        mu  = Hc.mean(axis=0)
        std = Hc.std(axis=0) + 1e-6
        D   = _mahalanobis_diag_batch(Hc, mu, std)
        tau = float(np.percentile(D, PROTO_PCT))
        proto[c] = {"mu": mu.tolist(), "std": std.tolist(), "tau": tau, "n": int(len(Hc))}
    return proto


**# 5) 전체 그룹 학습 실행**
- group_index.csv를 읽어 model/sensor 조합을 추출
- 각 조합에 대해 train_one_group 실행


In [ ]:
def list_available_groups(index_csv=f"{PROCESSED_DIR}/group_index.csv"):
    df = pd.read_csv(index_csv)
    pairs = sorted(df.groupby(["model","sensor"]).size().index.tolist())
    return pairs

pairs = list_available_groups()
print("학습 대상:", pairs)

# 각 조합 학습 (에폭/배치 크기 조정 가능)
for model_name, sensor in pairs:
    try:
        _ = train_one_group(model_name, sensor, epochs=10, batch_size=256, lr=3e-4)
    except Exception as e:
        print(f"[WARN] {model_name}/{sensor} 학습 중 예외:", e)


학습 대상: [('HH300', 'CLLBC'), ('HH300', 'CLYC'), ('SPRD', 'CLLBC'), ('SPRD', 'CSI')]
[WARN] HH300/CLLBC 학습 중 예외: train_one_group() missing 1 required positional argument: 'group_row'
[WARN] HH300/CLYC 학습 중 예외: train_one_group() missing 1 required positional argument: 'group_row'
[WARN] SPRD/CLLBC 학습 중 예외: train_one_group() missing 1 required positional argument: 'group_row'
[WARN] SPRD/CSI 학습 중 예외: train_one_group() missing 1 required positional argument: 'group_row'


**# 6) 추론 & 평가**
- 파일명으로 model/sensor 판단 → 해당 best.pt/scaler.json 로드
- 정규화/임계값/피크게이트 적용 → df_pred 반환
- 정확도/정밀도/재현율/F1/혼동행렬 계산


In [ ]:
# ==== 확률 기반 tri-class 설정 (CPS 사용 안함) ====
TRI_CLASS = True
TRI_POS_DEFAULT = 0.80   # Cs-137로 확정할 최소 확률
TRI_NEG_DEFAULT = 0.20   # BKG로 확정할 최대 확률
TRI_MARGIN = 0.15        # 저장된 best_threshold가 있을 때 여유폭

# (선택) Cs-137 포토피크 게이트를 추론에도 쓸지 여부
APPLY_PEAK_GATE = False   # ← 형상만 쓰려면 False 권장

def ensure_model_and_scaler(model_name, sensor):
    mdir = Path(MODELS_DIR) / safe_name(model_name) / safe_name(sensor)
    model_path  = mdir / "best.pt"
    scaler_path = mdir / "scaler.json"
    if not model_path.exists():
        raise FileNotFoundError(f"모델이 없습니다: {model_path}")
    if not scaler_path.exists():
        raise FileNotFoundError(f"스케일러가 없습니다: {scaler_path}")
    with open(scaler_path, "r") as f:
        scaler = json.load(f)
    return str(model_path), scaler

def decide_final_label_prob(prob, thr_pos, thr_neg, peak_ok=True):
    """CPS를 쓰지 않는 tri-class 결정"""
    if (prob >= thr_pos) and peak_ok:
        return 1, "Cs-137", "high_prob" if peak_ok else "peak_fail"
    elif prob <= thr_neg:
        return 0, "BKG", "low_prob"
    else:
        return 2, "Others", "uncertain"

def decide_final_label_prob_with_proto(prob, thr_pos, thr_neg,
                                       peak_ok, h_vec, proto):
    """
    확률 기반 tri-class + 임베딩 거리 기반 OOD 거절
    - proto: {"0":{"mu","std","tau"}, "1":{...}} (없으면 거리검정 생략)
    """
    # 피크 게이트 실패시 바로 Others(원하면 생략 가능)
    if APPLY_PEAK_GATE and (not peak_ok):
        return 2, "Others", "peak_fail"

    # 확률 만으로 1차 분류
    if prob >= thr_pos:
        first = 1
    elif prob <= thr_neg:
        first = 0
    else:
        return 2, "Others", "uncertain_prob"

    # 프로토타입 거리로 2차 검정 (멀면 Others)
    if isinstance(proto, dict) and str(first) in proto:
        mu  = np.array(proto[str(first)]["mu"], dtype=np.float32)
        std = np.array(proto[str(first)]["std"], dtype=np.float32)
        tau = float(proto[str(first)]["tau"])
        z = (h_vec - mu) / np.maximum(std, 1e-6)
        dist = float(np.sqrt((z*z).sum()))
        if dist > tau:
            return 2, "Others", f"far_from_class{first}(d={dist:.2f}>τ={tau:.2f})"
        else:
            return first, ("Cs-137" if first==1 else "BKG"), f"near_class{first}(d={dist:.2f}≤τ={tau:.2f})"
    else:
        # 프로토타입이 없으면 확률만으로
        return first, ("Cs-137" if first==1 else "BKG"), "no_proto"

def predict_file_auto_routed(txt_path: str, threshold=None, save_csv=True, plot_topk=0):
    model_name, sensor, serial = parse_filename(txt_path)
    model_path, scaler = ensure_model_and_scaler(model_name, sensor)

    # 모델 로드
    net, arch = load_cnn_from_ckpt_with_fallback(model_path, device)
    net.eval()
    print(f"[*] loaded model arch = {arch}, from = {model_path}")

    # 채널 통계/게이트/임계값/프로토타입
    chan_stats = None
    if scaler.get("spc_mean") is not None and scaler.get("spc_std") is not None:
        chan_stats = {
            "mean": np.array(scaler["spc_mean"], dtype=np.float32),
            "std":  np.array(scaler["spc_std"],  dtype=np.float32),
        }
        if "valid_mask" in scaler:
            chan_stats["valid_mask"] = np.array(scaler["valid_mask"], dtype=bool)

    proto = scaler.get("proto", None)
    peak_gate = scaler.get("peak_gate", None) if APPLY_PEAK_GATE else None

    # tri 임계 (best_threshold가 있으면 중앙값으로 사용)
    if threshold is None:
        base = float(scaler.get("best_threshold", 0.5))
        thr_pos = max(base, TRI_POS_DEFAULT)
        thr_neg = min(max(0.0, base - TRI_MARGIN), TRI_NEG_DEFAULT)
    else:
        thr_pos = float(threshold); thr_neg = 1.0 - thr_pos
    print(f"[*] tri thresholds: thr_pos={thr_pos:.3f}, thr_neg={thr_neg:.3f}, peak_gate={bool(peak_gate)} | proto={'yes' if proto else 'no'}")

    rows, kept, skipped = [], 0, 0
    with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            parsed = parse_line_strict_auto(line)
            if not parsed:
                skipped += 1
                continue
            ts, count, gc, temp_raw, spc_raw = parsed
            temp_c = temp_to_celsius(temp_raw)

            # 규칙 라벨(평가용) — 결정에는 사용 안 함
            rule_lbl_str, cps_val = make_label_from_spc(spc_raw, count)
            rule_lbl = label_to_id[rule_lbl_str]

            # 입력 전처리
            x_norm = normalize_spectrum(spc_raw, count, chan_stats)
            t_s = minmax_scale(temp_c, scaler["temp_c"]["min"], scaler["temp_c"]["max"])
            g_s = minmax_scale(gc,     scaler["gc"]["min"],     scaler["gc"]["max"])

            X   = torch.from_numpy(x_norm[None,None,:]).float().to(device)
            AUX = torch.tensor([[t_s, g_s]], dtype=torch.float32, device=device)
            with torch.no_grad():
                logit, h = net(X, AUX)
                prob = float(torch.sigmoid(logit).item())
                h_vec = h.detach().cpu().numpy().ravel()

            # (선택) 포토피크 창
            peak_ok = True
            if peak_gate:
                peak_ch = int(np.argmax(x_norm))
                c, tol = int(peak_gate["center"]), int(peak_gate["tol"])
                peak_ok = (abs(peak_ch - c) <= tol)

            # 최종 결정(형상 기반)
            final_id, final_str, reason = decide_final_label_prob_with_proto(
                prob, thr_pos, thr_neg, peak_ok, h_vec, proto
            )

            rows.append({
                "timestamp": ts, "model": model_name, "sensor": sensor, "serial": serial,
                "count": count, "gc": gc, "temp_c": temp_c, "cps": cps_val,  # 기록용
                "pred_prob": prob, "peak_ok": peak_ok, "thr_pos": thr_pos, "thr_neg": thr_neg,
                "final_label_id": final_id, "final_label_str": final_str, "final_reason": reason,
                "rule_label": rule_lbl, "rule_label_str": id_to_label[rule_lbl],
                "spc_raw": spc_raw.astype(np.float32), "spc_norm": x_norm.astype(np.float32),
            })
            kept += 1

    df_pred = pd.DataFrame(rows)
    print(f"[*] 라인 파싱: 사용 {kept}행, 제외 {skipped}행")

    if save_csv:
        out_csv = f"{PROCESSED_DIR}/pred_{Path(txt_path).stem}.csv"
        df_pred.to_csv(out_csv, index=False)
        print("저장:", out_csv)

    # 요약(평가용; Rule의 Cs-137 vs 나머지)
    y_true = df_pred["rule_label"].values
    y_pred_cs = (df_pred["final_label_str"].values == "Cs-137").astype(int)
    from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, accuracy_score
    acc = accuracy_score(y_true, y_pred_cs)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred_cs, average="binary", zero_division=0)
    cm = confusion_matrix(y_true, y_pred_cs, labels=[0,1])

    print(f"acc={acc:.3f} | prec={prec:.3f} | rec={rec:.3f} | f1={f1:.3f}")
    print("confusion_matrix [[TN,FP],[FN,TP]]:", cm.tolist())
    print("tri-class counts:", df_pred["final_label_str"].value_counts().to_dict())
    return df_pred



**# 7) 시각화**
- show_predictions_raw_and_norm(df_pred, k=…) : 한 라인당 RAW/NORM 두 그림
- launch_spectrum_browser(df_pred) : 좌/우 키/버튼/슬라이더로 탐색


In [ ]:
# ── 배치 표시
def show_predictions_raw_and_norm(df_pred: pd.DataFrame, k: int = 10, save_png: bool = False, out_dir: str = "/mnt/data/predictions/both_views"):
    from pathlib import Path
    if df_pred is None or len(df_pred)==0:
        raise ValueError("df_pred가 비었습니다.")
    need = ["timestamp","model","sensor","serial","cps","temp_c","gc","spc_raw","spc_norm"]
    miss = [c for c in need if c not in df_pred.columns]
    if miss: raise KeyError(f"필수 열 없음: {miss}")
    if save_png: Path(out_dir).mkdir(parents=True, exist_ok=True)

    n = min(k, len(df_pred))
    for i in range(n):
        r = df_pred.iloc[i]
        ttl_piece = f"pred={r.get('pred_label_str','?')} (p={float(r.get('pred_prob',np.nan)):.3f}) | rule={r.get('rule_label_str','?')}"
        # RAW
        x = np.asarray(r["spc_raw"], dtype=float)
        plt.figure(figsize=(9,3.2)); plt.plot(x)
        plt.title(f"RAW (cps) | {ttl_piece}")
        plt.xlabel("Channel (0..2047)"); plt.ylabel("Counts/sec"); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
        # NORM
        x = np.asarray(r["spc_norm"], dtype=float)
        plt.figure(figsize=(9,3.2)); plt.plot(x)
        plt.title(f"NORMALIZED (z-score) | {ttl_piece}")
        plt.xlabel("Channel (0..2047)"); plt.ylabel("Z-score"); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# ── 좌/우 탐색용 브라우저
import ipywidgets as widgets
from IPython.display import display, clear_output, Javascript

def _validate_df_pred(df: pd.DataFrame):
    if df is None or len(df)==0: raise ValueError("df_pred가 비었습니다.")
    need = ["timestamp","model","sensor","serial","cps","temp_c","gc","spc_raw","spc_norm"]
    miss = [c for c in need if c not in df.columns]
    if miss: raise KeyError(f"필수 열 없음: {miss}")

class InlineSpectrumBrowser:
    def __init__(self, df_pred: pd.DataFrame):
        _validate_df_pred(df_pred)
        self.df = df_pred.reset_index(drop=True)
        self.n  = len(self.df)
        self.i  = 0
        self.btn_prev = widgets.Button(description="⟵ 이전", tooltip="prev_key")
        self.btn_next = widgets.Button(description="다음 ⟶", tooltip="next_key")
        self.slider   = widgets.IntSlider(value=0, min=0, max=self.n-1, step=1, description="index", continuous_update=False)
        self.info_out = widgets.Output(layout={'border':'1px solid #ddd','padding':'6px'})
        self.btn_prev.on_click(lambda b: self.move(-1))
        self.btn_next.on_click(lambda b: self.move(+1))
        self.slider.observe(lambda ch: (ch["name"]=="value") and self._set_idx(int(ch["new"])), names="value")
        display(widgets.HBox([self.btn_prev, self.btn_next, self.slider, widgets.HTML("<b>키보드:</b> ⬅ / ➡ (이 셀 클릭)")]))
        display(self.info_out); self.redraw(); self._enable_arrow_keys_js()

    def _set_idx(self, v): self.i = v; self.redraw()
    def move(self, d): self.slider.value = (self.i + d) % self.n

    def redraw(self):
        r = self.df.iloc[self.i]
        with self.info_out:
            clear_output(wait=True)
            print(f"[{self.i+1}/{self.n}] {r['timestamp']} | {r['model']}/{r['sensor']} | Serial {r['serial']}")
            print(f"T={float(r['temp_c']):.1f}°C  GC={r['gc']}  CPS={float(r['cps']):.1f}")
            if "pred_label_str" in r and "pred_prob" in r: print(f"Prediction: {r['pred_label_str']} (p={float(r['pred_prob']):.3f})")
            if "rule_label_str" in r: print(f"Rule label : {r['rule_label_str']}")
            # RAW
            plt.figure(figsize=(9,3.2)); plt.plot(np.asarray(r["spc_raw"], float))
            plt.title(f"RAW (cps) | pred={r.get('pred_label_str','?')} (p={float(r.get('pred_prob',np.nan)):.3f}) | rule={r.get('rule_label_str','?')}")
            plt.xlabel("Channel (0..2047)"); plt.ylabel("Counts/sec"); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
            # NORM
            plt.figure(figsize=(9,3.2)); plt.plot(np.asarray(r["spc_norm"], float))
            plt.title(f"NORMALIZED (z-score) | pred={r.get('pred_label_str','?')} (p={float(r.get('pred_prob',np.nan)):.3f}) | rule={r.get('rule_label_str','?')}")
            plt.xlabel("Channel (0..2047)"); plt.ylabel("Z-score"); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

    def _enable_arrow_keys_js(self):
        js = Javascript(r"""
            (function(){
              if (window.__riid_cnn_arrow_binding__) return;
              window.__riid_cnn_arrow_binding__ = true;
              document.addEventListener('keydown', function(e){
                let tag = (e.target && e.target.tagName || '').toLowerCase();
                if (tag==='input'||tag==='textarea') return;
                if (e.key==='ArrowLeft'){
                  let btn=document.querySelector('button[title="prev_key"]'); if(btn){btn.click(); e.preventDefault();}
                } else if (e.key==='ArrowRight'){
                  let btn=document.querySelector('button[title="next_key"]'); if(btn){btn.click(); e.preventDefault();}
                }
              }, true);
            })();
        """); display(js)

def launch_spectrum_browser(df_pred: pd.DataFrame):
    _validate_df_pred(df_pred)
    df = df_pred.copy()
    if "final_label_str" not in df.columns:
        df["final_label_str"] = df.get("pred_label_str", "NA")

    class InlineSpectrumBrowserV2(InlineSpectrumBrowser):
        def redraw(self):
            r = self.df.iloc[self.i]
            with self.info_out:
                clear_output(wait=True)
                print(f"[{self.i+1}/{self.n}] {r['timestamp']} | {r['model']}/{r['sensor']} | Serial {r['serial']}")
                print(f"T={float(r['temp_c']):.1f}°C  GC={r['gc']}  CPS={float(r['cps']):.1f}  (참고용)")
                final_str = r.get("final_label_str","NA")
                prob = float(r.get("pred_prob", np.nan))
                thr_pos = r.get("thr_pos", np.nan); thr_neg = r.get("thr_neg", np.nan)
                print(f"Final: {final_str} (p={prob:.3f}, thr_pos={thr_pos:.2f}, thr_neg={thr_neg:.2f})"
                      f" | Rule: {r.get('rule_label_str','NA')} | peak_ok={r.get('peak_ok', True)}")

                plt.figure(figsize=(9,3.2))
                plt.plot(np.asarray(r["spc_raw"], float))
                plt.title(f"RAW (cps) | final={final_str} (p={prob:.3f}) | rule={r.get('rule_label_str','NA')}")
                plt.xlabel("Channel (0..2047)"); plt.ylabel("Counts/sec"); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

                plt.figure(figsize=(9,3.2))
                plt.plot(np.asarray(r["spc_norm"], float))
                plt.title(f"NORMALIZED (z-score) | final={final_str} (p={prob:.3f}) | rule={r.get('rule_label_str','NA')}")
                plt.xlabel("Channel (0..2047)"); plt.ylabel("Z-score"); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

                print(f"Final: {final_str} (p={prob:.3f}, thr_pos={thr_pos:.2f}, thr_neg={thr_neg:.2f})"
                      f" | Rule: {r.get('rule_label_str','NA')} | peak_ok={r.get('peak_ok', True)}"
                      f" | reason: {r.get('final_reason','')}")

    return InlineSpectrumBrowserV2(df)



**# 8) 실행 예시**
- test_file 전체 라인 추론 → 지표 출력
- 첫 10개 Raw/Norm 그래프
- 좌/우 탐색 브라우저


**## 진짜 핵종이 있는 테스트 데이터를 이용**

추론결과와 직접 비교

In [ ]:
# === 1) 테스트 라인에서 "[BackgroundThread]: [라벨]"까지 함께 파싱 ===
import re
import numpy as np
import pandas as pd
from pathlib import Path

NUC_RE = re.compile(
    r'^\[(?P<ts>[\d\-:\. ]+)\]\[BackgroundThread\]:\s*\[(?P<truth>[^\]]+)\]\s*'
    r'Count:\s*(?P<count>\d+),\s*GC:\s*(?P<gc>-?\d+),\s*Temperature:\s*(?P<temp>-?\d+),\s*Spc:\s*(?P<spc>.+)$'
)

def parse_line_with_truth(line: str):
    """
    예: [2025-09-20 03:00:00.623][BackgroundThread]: [BKG] Count: 60, GC: 33600, Temperature: 1126, Spc: ...
    반환: (timestamp, truth_label_str, count, gc, temp_raw, spc_np) or None
    """
    m = NUC_RE.match(line.strip())
    if not m:
        return None
    ts     = m.group("ts").strip()
    truth  = m.group("truth").strip()  # BKG, Cs-137, Co-57, Ba-133, Co-60 ...
    count  = int(m.group("count"))
    gc     = int(m.group("gc"))
    temp   = int(m.group("temp"))
    spc    = np.fromstring(m.group("spc"), sep=",", dtype=np.float32)
    return ts, truth, count, gc, temp, spc

def map_truth_to_trilabel(truth_str: str):
    """
    tri-class(3분류)용 진짜 라벨 매핑
    - 'BKG' → 'BKG'
    - 'Cs-137' (대소문자/하이픈 변형 포함) → 'Cs-137'
    - 나머지(Co-57, Ba-133, Co-60 등) → 'Others'
    """
    s = truth_str.upper().replace(" ", "")
    if s in ("BKG", "BACKGROUND", "BG"):
        return "BKG"
    if ("CS137" in s) or ("CS-137" in s) or ("CS_137" in s):
        return "Cs-137"
    return "Others"

TRI_CLASS_ORDER = ["BKG", "Cs-137", "Others"]
TRI_TO_ID = {k:i for i,k in enumerate(TRI_CLASS_ORDER)}

# === 2) 브라우저 출력에서 진짜 라벨을 보여주도록 보강(이미 함수가 있다면 출력만 교체) ===
def _fmt_head_line(r):
    # r: df_pred의 한 행(dict)
    head = (f"[{r.get('idx_show','?')}] {r['timestamp']} | {r['model']}/{r['sensor']} | Serial {r['serial']}\n"
            f"T={r['temp_c']:.1f}°C  GC={r['gc']}  CPS={r['cps']:.1f}\n")
    mid  = (f"Final: {r['final_label_str']} (p={r['pred_prob']:.3f}) | "
            f"Truth: {r.get('truth_isotope','NA')} → {r.get('truth_tri','NA')}\n"
            f"Reason: {r.get('final_reason','')}")
    return head + mid

# === 3) 추론 함수 패치: 진짜 라벨 비교로 평가(3분류 + Cs-137 이진) ===
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, accuracy_score

def predict_file_auto_routed(txt_path: str, threshold=None, save_csv=True, plot_topk=0):
    # (1) 어느 그룹 모델을 쓸지 자동 라우팅
    model_name, sensor, serial = parse_filename(txt_path)
    model_path, scaler = ensure_model_and_scaler(model_name, sensor)

    # (2) 모델 로드(v2↔v1 호환)
    net, arch = load_cnn_from_ckpt_with_fallback(model_path, device)
    net.eval()
    print(f"[*] loaded model arch = {arch}, from = {model_path}")

    # (3) 전처리 리소스
    chan_stats = None
    if scaler.get("spc_mean") is not None and scaler.get("spc_std") is not None:
        chan_stats = {
            "mean": np.array(scaler["spc_mean"], dtype=np.float32),
            "std":  np.array(scaler["spc_std"],  dtype=np.float32),
        }
        if "valid_mask" in scaler:
            chan_stats["valid_mask"] = np.array(scaler["valid_mask"], dtype=bool)

    proto = scaler.get("proto", None)
    peak_gate = scaler.get("peak_gate", None) if APPLY_PEAK_GATE else None

    # tri 임계 (best_threshold 기반)
    if threshold is None:
        base = float(scaler.get("best_threshold", 0.5))
        thr_pos = max(base, TRI_POS_DEFAULT)
        thr_neg = min(max(0.0, base - TRI_MARGIN), TRI_NEG_DEFAULT)
    else:
        thr_pos = float(threshold); thr_neg = 1.0 - thr_pos
    print(f"[*] tri thresholds: thr_pos={thr_pos:.3f}, thr_neg={thr_neg:.3f}, peak_gate={bool(peak_gate)} | proto={'yes' if proto else 'no'}")

    rows, kept, skipped = [], 0, 0

    with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
        for i, line in enumerate(f):
            parsed = parse_line_with_truth(line)
            if not parsed:
                skipped += 1
                continue
            ts, truth_iso, count, gc, temp_raw, spc_raw = parsed
            temp_c = temp_to_celsius(temp_raw)

            # ── 입력 전처리
            x_norm = normalize_spectrum(spc_raw, count, chan_stats)
            t_s = minmax_scale(temp_c, scaler["temp_c"]["min"], scaler["temp_c"]["max"])
            g_s = minmax_scale(gc,     scaler["gc"]["min"],     scaler["gc"]["max"])

            X   = torch.from_numpy(x_norm[None,None,:]).float().to(device)
            AUX = torch.tensor([[t_s, g_s]], dtype=torch.float32, device=device)
            with torch.no_grad():
                logit, h = net(X, AUX)
                prob = float(torch.sigmoid(logit).item())
                h_vec = h.detach().cpu().numpy().ravel()

            # ── (선택) 포토피크 창 일치성
            peak_ok = True
            if peak_gate:
                peak_ch = int(np.argmax(x_norm))
                c, tol = int(peak_gate["center"]), int(peak_gate["tol"])
                peak_ok = (abs(peak_ch - c) <= tol)

            # ── 최종 결정(확률+프로토타입 거리)
            final_id, final_str, reason = decide_final_label_prob_with_proto(
                prob, thr_pos, thr_neg, peak_ok, h_vec, proto
            )

            # ── 진짜 라벨(텍스트 → tri-class)
            truth_tri = map_truth_to_trilabel(truth_iso)
            truth_id  = TRI_TO_ID[truth_tri]

            rows.append({
                "idx_show": f"{i+1}",
                "timestamp": ts, "model": model_name, "sensor": sensor, "serial": serial,
                "count": count, "gc": gc, "temp_c": temp_c,
                "cps": spc_raw.sum()/max(count,1e-6),  # 기록용(결정에는 미사용)
                "pred_prob": prob, "peak_ok": peak_ok, "thr_pos": thr_pos, "thr_neg": thr_neg,
                "final_label_id": final_id, "final_label_str": final_str, "final_reason": reason,
                "truth_isotope": truth_iso, "truth_tri": truth_tri, "truth_id": truth_id,
                "spc_raw": spc_raw.astype(np.float32), "spc_norm": x_norm.astype(np.float32),
            })
            kept += 1

    df_pred = pd.DataFrame(rows)
    print(f"[+] 라인 파싱: 사용 {kept}행, 제외 {skipped}행")

    # ── 평가 (1) 3분류: Truth(BKG/Cs-137/Others) vs Pred(BKG/Cs-137/Others)
    y_true_tri = df_pred["truth_id"].values
    y_pred_tri = df_pred["final_label_id"].values
    tri_acc = accuracy_score(y_true_tri, y_pred_tri)
    tri_cm  = confusion_matrix(y_true_tri, y_pred_tri, labels=[0,1,2])

    # ── 평가 (2) 이진: Cs-137 vs 나머지
    y_true_bin = (df_pred["truth_tri"].values == "Cs-137").astype(int)
    y_pred_bin = (df_pred["final_label_str"].values == "Cs-137").astype(int)
    acc = accuracy_score(y_true_bin, y_pred_bin)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true_bin, y_pred_bin, average="binary", zero_division=0
    )
    cm_bin = confusion_matrix(y_true_bin, y_pred_bin, labels=[0,1])

    # ── 출력
    print(f"tri-acc={tri_acc:.3f} | tri-class counts={df_pred['final_label_str'].value_counts().to_dict()}")
    print("tri-confusion (rows=True cols=Pred) order [BKG, Cs-137, Others]:")
    print(tri_cm)

    print("\nBinary (Cs-137 vs rest): "
          f"acc={acc:.3f} | prec={prec:.3f} | rec={rec:.3f} | f1={f1:.3f}")
    print("binary-confusion [[TN,FP],[FN,TP]]:")
    print(cm_bin)

    # 표 객체도 같이 넘겨줌(노트북에서 보기 좋게)
    df_metrics_tri = pd.DataFrame(
        {"Class": TRI_CLASS_ORDER,
         "Support(True)": [int((y_true_tri==i).sum()) for i in range(3)],
         "Pred as BKG":   tri_cm[:,0].tolist(),
         "Pred as Cs-137":tri_cm[:,1].tolist(),
         "Pred as Others":tri_cm[:,2].tolist()}
    )
    df_metrics_bin = pd.DataFrame([{
        "Accuracy": acc, "Precision": prec, "Recall": rec, "F1-score": f1
    }])

    if save_csv:
        out_csv = f"{PROCESSED_DIR}/pred_{Path(txt_path).stem}.csv"
        df_pred.to_csv(out_csv, index=False)
        print("저장:", out_csv)

    # 반환: 예측 결과 + 표 2개(원하면 display(df_)로 보기)
    return df_pred, df_metrics_tri, df_metrics_bin

# === 4) 브라우저 함수의 헤더 출력만 'Truth'로 교체 (이미 정의돼있으면 이 라인만 수정) ===
def launch_spectrum_browser(df_pred):
    import matplotlib.pyplot as plt
    from ipywidgets import VBox, HBox, Button, IntSlider, Output, Label

    N = len(df_pred)
    idx = 0
    out = Output()

    def draw(i):
        r = df_pred.iloc[i].to_dict()
        r["idx_show"] = f"{i+1}/{N}"
        with out:
            out.clear_output(wait=True)
            print(_fmt_head_line(r))
            # --- 그래프(생략 없이 기존 구현 재사용) ---
            fig = plt.figure(figsize=(9,6))
            ax1 = fig.add_subplot(2,1,1)
            ax1.plot(r["spc_raw"]); ax1.set_title(f"RAW (cps) | pred={r['final_label_str']} (p={r['pred_prob']:.3f}) | truth={r['truth_tri']}")
            ax1.set_ylabel("Counts/sec"); ax1.set_xlabel("Channel (0..2047)")
            ax2 = fig.add_subplot(2,1,2)
            ax2.plot(r["spc_norm"]); ax2.set_title("NORMALIZED (z-score)")
            ax2.set_ylabel("Z-score"); ax2.set_xlabel("Channel (0..2047)")
            plt.tight_layout(); plt.show()

    # 위젯
    btn_prev = Button(description="← 이전")
    btn_next = Button(description="다음 →")
    slider   = IntSlider(value=0, min=0, max=max(0,N-1), step=1, description="index")

    def on_prev(_):
        nonlocal idx; idx = (idx-1) % N; slider.value = idx; draw(idx)
    def on_next(_):
        nonlocal idx; idx = (idx+1) % N; slider.value = idx; draw(idx)
    def on_slide(change):
        nonlocal idx; idx = change["new"]; draw(idx)

    btn_prev.on_click(on_prev); btn_next.on_click(on_next)
    slider.observe(on_slide, names="value")

    ui = VBox([HBox([btn_prev, btn_next, slider]), out])
    display(ui)
    if N>0: draw(0)


2) 평가지표 표로 표시 (정확도/정밀도/재현율/F1 + 혼동행렬)

In [ ]:
test_file = "/content/drive/MyDrive/RIID_CNN/TestData/hwSpc_PR25ZY-G(CSI).txt"

# 추론 + 평가(진짜 라벨 기준)
df_pred, tri_table, bin_table = predict_file_auto_routed(test_file, threshold=None, save_csv=True)

# 표로 보기
display(tri_table)   # 3분류 혼동표(클래스별 예측 분포)
display(bin_table)   # Cs-137 vs 나머지 이진 지표

# 좌↔우 탐색 브라우저 (헤더에 Truth/Pred 모두 표기)
launch_spectrum_browser(df_pred)


[*] loaded model arch = v2, from = /content/drive/MyDrive/RIID_CNN/processed/models/SPRD/CSI/best.pt
[*] tri thresholds: thr_pos=0.800, thr_neg=0.200, peak_gate=False | proto=no
[+] 라인 파싱: 사용 90행, 제외 0행
tri-acc=0.189 | tri-class counts={'BKG': 68, 'Cs-137': 22}
tri-confusion (rows=True cols=Pred) order [BKG, Cs-137, Others]:
[[14  0  0]
 [ 0  3  0]
 [54 19  0]]

Binary (Cs-137 vs rest): acc=0.789 | prec=0.136 | rec=1.000 | f1=0.240
binary-confusion [[TN,FP],[FN,TP]]:
[[68 19]
 [ 0  3]]
저장: /content/drive/MyDrive/RIID_CNN/processed/pred_hwSpc_PR25ZY-G(CSI).csv


,Class,Support(True),Pred as BKG,Pred as Cs-137,Pred as Others
0,BKG,14,14,0,0
1,Cs-137,3,0,3,0
2,Others,73,54,19,0


,Accuracy,Precision,Recall,F1-score
0,0.788889,0.136364,1.0,0.24
